In [ ]:
!pip install fuzzywuzzy
!pip install python-Levenshtein
!pip install lxml 
!pip install html5lib 

In [1]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import pandas as pd

In [29]:
salary_data = pd.read_html('https://hoopshype.com/salaries/players/2022-2023/')[0]

In [30]:
player_data = pd.read_csv('player_data_2022.csv')

In [31]:
def fuzzy_merge(df_1, df_2, key1, key2, threshold=90, limit=2):
    """
    :param df_1: the left table to join
    :param df_2: the right table to join
    :param key1: key column of the left table
    :param key2: key column of the right table
    :param threshold: how close the matches should be to return a match, based on Levenshtein distance
    :param limit: the amount of matches that will get returned, these are sorted high to low
    :return: dataframe with boths keys and matches
    """
    s = df_2[key2].tolist()
    
    m = df_1[key1].apply(lambda x: process.extract(x, s, limit=limit))    
    df_1['matches'] = m
    
    m2 = df_1['matches'].apply(lambda x: ', '.join([i[0] for i in x if i[1] >= threshold]))
    df_1['matches'] = m2
    
    return df_1

In [32]:
overlap = fuzzy_merge(player_data, salary_data, 'Name', 'Player', threshold=80, limit=1)

In [33]:
overlap

,Name,Position,Age,Team,Games,Minutes,3P,2P,FT,Rebounds,Assists,Steals,Blocks,Turnovers,Fouls,Points,matches
0,AJ Griffin,SF,19,ATL,72,1401,101,147,42,153,73,42,12,42,87,639,AJ Griffin
1,Blake Wesley,SG,19,SAS,37,669,20,49,26,81,99,25,5,65,67,184,Blake Wesley
2,Dominick Barlow,PF,19,SAS,28,408,0,46,18,102,24,10,19,15,56,110,Dominick Barlow
3,Dyson Daniels,PG,19,NOP,59,1042,27,60,26,188,134,43,11,57,99,227,Dyson Daniels
4,Jabari Smith Jr.,PF,19,HOU,79,2451,120,244,162,569,101,43,74,104,227,1010,Jabari Smith
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
533,P.J. Tucker,PF,37,PHI,75,1920,55,41,19,295,60,39,15,44,180,266,PJ Tucker
534,Taj Gibson,C,37,WAS,49,480,8,57,30,93,34,15,12,26,84,168,Taj Gibson
535,LeBron James,PF,38,LAL,55,1954,121,488,251,457,375,50,32,178,88,1590,LeBron James
536,Andre Iguodala,PF,39,GSW,8,113,1,6,2,17,19,4,3,9,11,17,Andre Iguodala


In [34]:
total = pd.merge(salary_data, overlap, left_on='Player', right_on='matches').drop(columns = ['Unnamed: 0', '2022/23(*)', 'Name', 'matches']).rename(columns={"2022/23":"Salary"})
total.to_csv('total.csv', index=False)

In [35]:
total

,Player,Salary,Position,Age,Team,Games,Minutes,3P,2P,FT,Rebounds,Assists,Steals,Blocks,Turnovers,Fouls,Points
0,Stephen Curry,"$48,070,014",PG,34,GSW,56,1941,273,286,257,341,352,52,20,179,117,1648
1,John Wall,"$47,345,760",PG,32,LAC,34,755,33,105,77,92,178,27,12,80,59,386
2,Russell Westbrook,"$47,080,179",PG,34,TOT,73,2126,89,343,206,423,551,76,33,255,162,1159
3,LeBron James,"$44,474,988",PF,38,LAL,55,1954,121,488,251,457,375,50,32,178,88,1590
4,Kevin Durant,"$44,119,845",PF,34,TOT,47,1672,93,390,307,313,235,34,67,156,99,1366
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,Justin Minaya,"$35,096",SF,23,POR,4,89,3,4,0,15,4,2,5,4,9,17
532,Kobi Simmons,"$32,795",SG,25,CHO,5,28,1,0,2,4,5,0,2,1,0,5
533,Gabe York,"$32,171",SG,29,IND,3,56,6,2,2,6,5,2,0,0,5,24
534,RaiQuan Gray,"$5,849",PF,23,BRK,1,35,2,4,2,9,7,0,1,4,5,16


In [37]:
total['Salary'] = total['Salary'].str.replace(',', '').str.replace('$', '').astype(int)
total

,Player,Salary,Position,Age,Team,Games,Minutes,3P,2P,FT,Rebounds,Assists,Steals,Blocks,Turnovers,Fouls,Points
0,Stephen Curry,48070014,PG,34,GSW,56,1941,273,286,257,341,352,52,20,179,117,1648
1,John Wall,47345760,PG,32,LAC,34,755,33,105,77,92,178,27,12,80,59,386
2,Russell Westbrook,47080179,PG,34,TOT,73,2126,89,343,206,423,551,76,33,255,162,1159
3,LeBron James,44474988,PF,38,LAL,55,1954,121,488,251,457,375,50,32,178,88,1590
4,Kevin Durant,44119845,PF,34,TOT,47,1672,93,390,307,313,235,34,67,156,99,1366
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,Justin Minaya,35096,SF,23,POR,4,89,3,4,0,15,4,2,5,4,9,17
532,Kobi Simmons,32795,SG,25,CHO,5,28,1,0,2,4,5,0,2,1,0,5
533,Gabe York,32171,SG,29,IND,3,56,6,2,2,6,5,2,0,0,5,24
534,RaiQuan Gray,5849,PF,23,BRK,1,35,2,4,2,9,7,0,1,4,5,16


In [39]:
total[['Player', 'Salary']].to_csv('salary_data_2022.csv', index=False)

In [43]:
total.drop(columns=['Salary']).sort_values(by='Points', ascending=False).to_csv('player_data_2022.csv', index=False)